In [119]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     
    return pd.read_sql(sql, con)

In [120]:
def run_sql_file(path):
    with open(path, 'r') as f:
        script = f.read()

    con.executescript(script)
    con.commit()

    print("Run:", path)

In [121]:
run_sql_file(r"C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\01_revenue_cutomer.sql")

Run: C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\01_revenue_cutomer.sql


In [122]:
q("""select
count(*)as total_customers,
round(avg(arpu_inr),2)as average_arpu,
count(distinct arpu_inr)as distinct_arpu_values
from customers;
""")

,total_customers,average_arpu,distinct_arpu_values
0,22000,421.65,13


In [123]:
q("""SELECT
COUNT(*) AS customers_without_payments
FROM customers AS c
WHERE NOT EXISTS (
SELECT 1
FROM recharges AS r
WHERE r.customer_id = c.customer_id
AND r.recharge_date >= '2025-01-01'
AND r.recharge_date < '2025-07-01');""")

,customers_without_payments
0,3711


In [124]:
q("""
select name,type
from sqlite_master
where name = 'v_revenue_customer';
""")

,name,type
0,v_revenue_customer,view


In [125]:
q("""
select * from v_revenue_customer;
""")

,customer_id,recharge_count,recharge_total_6m,actual_arpu_6m
0,1,3,897,149.50
1,2,2,198,33.00
2,3,6,5073,845.50
3,4,0,0,0.00
4,5,2,1438,239.67
...,...,...,...,...
21995,21996,6,10478,1746.33
21996,21997,1,719,119.83
21997,21998,5,1045,174.17
21998,21999,2,798,133.00


In [126]:
q("""SELECT
SUM(CASE WHEN recharge_count > 0 THEN 1 ELSE 0 END) AS customers_with_payments,
SUM(CASE WHEN recharge_count = 0 THEN 1 ELSE 0 END) AS customers_without_payments
FROM v_revenue_customer;""")

,customers_with_payments,customers_without_payments
0,18289,3711


In [127]:
run_sql_file(r"C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\02_usage_customer.sql")

Run: C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\02_usage_customer.sql


In [128]:
q("""SELECT * FROM V_usage_customer;""")

,customer_id,usage_days,total_data_mb,total_voice_min,total_sms,roaming_days,mb_per_active_day,mb_per_observed_day,last_usage_date,recency_days
0,1,17,12405.0,333,3,0,729.71,68.54,2025-04-26,65.0
1,2,11,2822.5,424,6,1,256.59,15.59,2025-02-11,139.0
2,3,55,102791.3,2205,23,1,1868.93,567.91,2025-06-25,5.0
3,4,40,8755.2,2262,26,0,218.88,48.37,2025-03-16,106.0
4,5,39,44725.0,1364,9,0,1146.79,247.10,2025-06-18,12.0
...,...,...,...,...,...,...,...,...,...,...
21995,21996,79,522264.4,4713,51,2,6610.94,2885.44,2025-06-30,0.0
21996,21997,101,141403.8,6621,83,2,1400.04,781.24,2025-06-28,2.0
21997,21998,65,4881.9,2111,30,6,75.11,26.97,2025-06-30,0.0
21998,21999,18,20771.6,596,11,0,1153.98,114.76,2025-03-31,91.0


In [129]:
q("""select count(*) as customers_with_complaints 
from customers as c 
left join complaints as cp on c.customer_id = cp.customer_id 
where cp.customer_id is null;""")

,customers_with_complaints
0,6631


In [130]:
run_sql_file(r'C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\03_complaints_customer.sql')

Run: C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\03_complaints_customer.sql


In [131]:
q("""select count(*) from v_complaint_customer;""")

,count(*)
0,22000


In [132]:
q("""select * from v_complaint_customer;""")

,customer_id,complaint_count,unresolved_count,resolved_count,avg_resolution_days
0,1,1,1,0,NaN
1,2,1,1,0,NaN
2,3,0,0,0,NaN
3,4,0,0,0,NaN
4,5,2,1,1,13.0
...,...,...,...,...,...
21995,21996,2,1,1,11.0
21996,21997,1,0,1,13.0
21997,21998,1,0,1,10.0
21998,21999,0,0,0,NaN


In [133]:
q("""select count(*) from v_complaint_customer where complaint_count = 0;""")

,count(*)
0,6631


In [134]:
run_sql_file(r'C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\05_usage_month.sql')

Run: C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\05_usage_month.sql


In [135]:
q("""select * from v_usage_month;""")

,customer_id,month,usage_days,total_mb,total_voice,roaming_days
0,1,2025-01,2,1222.5,60,0
1,1,2025-02,4,4496.3,114,0
2,1,2025-03,5,2871.0,50,0
3,1,2025-04,6,3815.2,109,0
4,2,2025-01,7,1870.5,225,1
...,...,...,...,...,...,...
117827,22000,2025-02,20,29489.8,1435,2
117828,22000,2025-03,20,33024.3,1489,0
117829,22000,2025-04,17,23536.9,1170,1
117830,22000,2025-05,23,39269.1,1521,1


In [137]:
run_sql_file(r'C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\04_plan_clean.sql')

Run: C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\04_plan_clean.sql


In [138]:
run_sql_file(r'C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\06_customer_features.sql')

Run: C:\Users\muham\OneDrive\Documents\airtel telicom analysis\sql\06_customer_features.sql


In [139]:
q("""SELECT COUNT(*) AS rows_,
COUNT(DISTINCT customer_id) AS subs,
SUM(usage_days = 0) AS no_usage,
SUM(recharge_count = 0) AS no_payment,
SUM(complaint_count = 0) AS never_complained,
ROUND(AVG(tenure_days), 1) AS tenure,
ROUND(AVG(actual_arpu_6m), 2) AS actual_arpu,
ROUND(100.0 * AVG(is_churned), 2) AS churn
FROM customer_features;""")

,rows_,subs,no_usage,no_payment,never_complained,tenure,actual_arpu,churn
0,22000,22000,4,3711,6631,697.6,344.46,23.75
